# Módulo 00 · Un sistema RAG mínimo

Este cuaderno construye el sistema RAG más simple posible, sin dependencias ni
credenciales, para mostrar sus tres etapas:

1. **Recuperación** (*retrieval*): localizar la información relevante.
2. **Aumento** (*augmentation*): incorporarla al prompt.
3. **Generación** (*generation*): pedir al modelo que redacte la respuesta (se implementa en el módulo 07).

## Base de conocimiento

En un sistema real habría miles de documentos. Para ilustrar el concepto bastan cinco
frases de NovaTech, la empresa ficticia que se utiliza como caso de estudio.

In [1]:
knowledge_base = [
    "En NovaTech cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de diciembre.",
    "Las vacaciones se solicitan en NovaHR con al menos 15 días de antelación.",
    "La dieta diaria para viajes de trabajo en España es de 45 €.",
    "Si pierdes el portátil o sufres un robo, avisa a la guardia de IT en el 900 000 111.",
    "El plan Starter de NovaDesk cuesta 15 € por agente y mes.",
]

## Etapa 1: recuperación

La estrategia más elemental consiste en contar cuántas palabras comparte la pregunta
con cada documento, tras descartar las palabras vacías (*stopwords*) como "el" o "de".

In [2]:
import re

STOPWORDS = set(
    "a al con cuál cuántos de del el en es la las lo los me mi para por qué se si tengo un una y".split()
)


def tokenize(text: str) -> set[str]:
    return {word for word in re.findall(r"\w+", text.lower()) if word not in STOPWORDS}


def search(question: str, documents: list[str], k: int = 2) -> list[str]:
    question_words = tokenize(question)
    scored = [(len(question_words & tokenize(doc)), doc) for doc in documents]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [doc for score, doc in scored[:k] if score > 0]


question = "¿Cuántos días de vacaciones tengo?"
print("Términos de búsqueda:", sorted(tokenize(question)))
for doc in search(question, knowledge_base):
    print("-", doc)

Términos de búsqueda: ['días', 'vacaciones']
- En NovaTech cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de diciembre.
- Las vacaciones se solicitan en NovaHR con al menos 15 días de antelación.


## Etapa 2: aumento

La pregunta y los fragmentos recuperados se combinan en un único prompt. Las
instrucciones exigen al modelo que se limite al contexto y que reconozca cuándo no
dispone de la información, lo que reduce de forma notable las respuestas inventadas.

In [3]:
def build_prompt(question: str, context: list[str]) -> str:
    context_text = "\n".join(f"- {fragment}" for fragment in context) or "(sin contexto)"
    return f"""Usted es el asistente interno de NovaTech.
Responda a la pregunta utilizando SOLO la información del contexto.
Si el contexto no contiene la respuesta, indique "No dispongo de esa información".

Contexto:
{context_text}

Pregunta: {question}
Respuesta:"""


print(build_prompt(question, search(question, knowledge_base)))

Usted es el asistente interno de NovaTech.
Responda a la pregunta utilizando SOLO la información del contexto.
Si el contexto no contiene la respuesta, indique "No dispongo de esa información".

Contexto:
- En NovaTech cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de diciembre.
- Las vacaciones se solicitan en NovaHR con al menos 15 días de antelación.

Pregunta: ¿Cuántos días de vacaciones tengo?
Respuesta:


## Etapa 3: generación

Este prompt es lo que recibiría el modelo de lenguaje. Con el contexto delante, la
respuesta esperada sería:

> Dispone de 23 días laborables de vacaciones al año, además del 24 y el 31 de
> diciembre. Deben solicitarse en NovaHR con al menos 15 días de antelación.

Lo relevante es que **el modelo no necesita conocer NovaTech de antemano**: la
información necesaria viaja en el prompt.

## Preguntas sin respuesta en la documentación

Si la búsqueda no encuentra nada, el contexto queda vacío y las instrucciones indican
al modelo que se abstenga en lugar de inventar.

In [4]:
question = "¿Cuál es el salario de la dirección general?"
print(build_prompt(question, search(question, knowledge_base)))

Usted es el asistente interno de NovaTech.
Responda a la pregunta utilizando SOLO la información del contexto.
Si el contexto no contiene la respuesta, indique "No dispongo de esa información".

Contexto:
(sin contexto)

Pregunta: ¿Cuál es el salario de la dirección general?
Respuesta:


## Limitaciones de este enfoque

La siguiente pregunta tiene respuesta en la base de conocimiento (la frase sobre el
portátil), pero la búsqueda no la encuentra:

In [5]:
question = "¿Qué pasa si me roban el ordenador?"
print("Términos de búsqueda:", sorted(tokenize(question)))
print("Resultados:", search(question, knowledge_base))

Términos de búsqueda: ['ordenador', 'pasa', 'roban']
Resultados: []


"roban" no coincide con "robo", ni "ordenador" con "portátil": la búsqueda solo
reconoce palabras idénticas, no **significados**. El resto del programa resuelve
estas limitaciones de forma progresiva:

| Limitación | Módulo |
|---|---|
| Los documentos reales son extensos y deben dividirse | 02 · Segmentación |
| Todas las palabras pesan lo mismo y las variantes no coinciden | 03 · Búsqueda léxica (BM25) |
| No se reconocen sinónimos ni paráfrasis | 04 · Embeddings |
| La búsqueda debe ser eficiente sobre miles de fragmentos | 05 · Almacén vectorial |
| Falta la generación real de la respuesta, con citas | 07 · Generación |